In [1]:
import sys
print("python exe:", sys.executable)   # full path to the interpreter
print("python ver:", sys.version)

# try torch only in the working notebook
try:
    import torch
    print("torch      :", torch.__version__, torch.__file__)
except ModuleNotFoundError as e:
    print("torch not importable:", e)



python exe: c:\Users\aneek\anaconda3\envs\tf_gpu_env\python.exe
python ver: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch      : 1.12.1+cu113 c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\torch\__init__.py


In [2]:
import tensorflow as tf
print(tf.__version__)  # This should print the version of TensorFlow
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

print("CUDA version:", tf.sysconfig.get_build_info()["cuda_version"])
print("cuDNN version:", tf.sysconfig.get_build_info()["cudnn_version"])

from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

2.10.0
Num GPUs Available:  1
CUDA version: 64_112
cuDNN version: 64_8
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 15506433069630362548
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5713690624
locality {
  bus_id: 1
  links {
  }
}
incarnation: 3333193304646215065
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


In [3]:
import time
import tensorflow as tf
import psutil

class PowerMonitor:
    def __init__(self):
        self.gpu_available = tf.config.list_physical_devices('GPU')
        
        # Hardware power specifications (adjust these values for your system)
        self.cpu_tdp = 65    # Typical TDP for desktop CPUs in watts
        self.gpu_tdp = 250   # Typical TDP for desktop GPUs in watts
        
    def get_stats(self):
        """Get system stats with power estimation"""
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'ram_mb': psutil.virtual_memory().used / (1024**2),
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85  # Base CPU power
        }
        
        if self.gpu_available:
            try:
                # TensorFlow GPU memory monitoring
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                stats.update({
                    'gpu_mem_mb': mem_info['current'] / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75  # Add GPU power estimate
                })
            except:
                pass
                
        return stats

# Initialize monitor
monitor = PowerMonitor()

In [4]:
import time
import torch
import psutil
import os

class PowerMonitor1:
    def __init__(self):
        self.gpu_available = torch.cuda.is_available()
        self.process = psutil.Process(os.getpid())  # Track current process
        
        # Hardware power specifications
        self.cpu_tdp = 65
        self.gpu_tdp = 250
        
    def get_stats(self):
        """Get process-specific stats with power estimation"""
        process_memory = self.process.memory_info()
        
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'process_ram_mb': process_memory.rss / (1024**2),  # Only this process's RAM
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85
        }
        
        if self.gpu_available:
            try:
                gpu_memory_allocated = torch.cuda.memory_allocated()
                stats.update({
                    'gpu_mem_mb': gpu_memory_allocated / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75
                })
            except Exception as e:
                print(f"Error retrieving GPU memory: {e}")
                
        return stats

# Initialize monitor1
monitor1 = PowerMonitor1()

# Model

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from PIL import Image
from transformers import AutoImageProcessor, Swinv2ForImageClassification, Swinv2Config


# Load Swin Transformer V2 model and processor for image classification
processor = AutoImageProcessor.from_pretrained("microsoft/swinv2-tiny-patch4-window8-256")
config = Swinv2Config.from_pretrained("microsoft/swinv2-tiny-patch4-window8-256")

# Modify for binary classification
config.num_labels = 2

# Load model and update classifier head
model = Swinv2ForImageClassification.from_pretrained(
    "microsoft/swinv2-tiny-patch4-window8-256",
    config=config,
    ignore_mismatched_sizes=True
)
model.classifier = nn.Linear(model.classifier.in_features, config.num_labels)



# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Dataset class
class ImageClassificationDataset(Dataset):
    def __init__(self, images, labels, processor):
        self.images = images
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)
        elif isinstance(image, str):
            image = Image.open(image).convert("RGB")

        inputs = self.processor(images=image, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in inputs.items()}, torch.tensor(label)

c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration. Please open a PR/issue to update `preprocessor_config.json` to use `image_processor_type` instead of `feature_extractor_type`. This warning will be removed in v4.40.
Some weights of Swinv2ForImageClassification were not initialized from the model checkpoint at microsoft/swinv2-tiny-patch4-window8-256 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.S

In [6]:
# ============================================================
# VALIDATION FUNCTION
# ============================================================

def evaluate_epoch(
    model,
    data_loader,
    device
):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.inference_mode():

        for inputs, labels in data_loader:

            inputs = {
                key: value.to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )
                for key, value in inputs.items()
            }

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            # Supplying labels makes the Hugging Face model
            # calculate CrossEntropyLoss automatically.
            outputs = model(
                **inputs,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = labels.size(0)

            total_loss += (
                loss.item()
                * current_batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += current_batch_size

    validation_loss = (
        total_loss / total_samples
    )

    validation_accuracy = (
        correct_predictions / total_samples
    )

    return (
        validation_loss,
        validation_accuracy
    )
# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=10
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):

        model.train()

        total_train_loss = 0.0
        correct_train_predictions = 0
        total_train_samples = 0

        for inputs, labels in train_loader:

            inputs = {
                key: value.to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )
                for key, value in inputs.items()
            }

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(
                **inputs,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            loss.backward()
            optimizer.step()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            current_batch_size = labels.size(0)

            total_train_loss += (
                loss.item()
                * current_batch_size
            )

            correct_train_predictions += (
                predictions == labels
            ).sum().item()

            total_train_samples += (
                current_batch_size
            )

        train_loss = (
            total_train_loss
            / total_train_samples
        )

        train_accuracy = (
            correct_train_predictions
            / total_train_samples
        )

        val_loss, val_accuracy = evaluate_epoch(
            model=model,
            data_loader=val_loader,
            device=device
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

    return history
optimizer = optim.Adam(model.parameters(),lr=1e-4)

In [7]:
# ============================================================
# COMPLETE SWINV2 EVALUATION FUNCTION
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc
)


def evaluate_complete(
    model,
    loader
):
    """
    Complete binary classification evaluation.

    Label mapping:
        0 = real
        1 = fake
    """

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []
    all_fake_probabilities = []

    with torch.inference_mode():

        for batch in loader:

            # Your dataset returns:
            # inputs dictionary, labels tensor
            inputs, labels = batch

            inputs = {
                key: value.to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )
                for key, value in inputs.items()
            }

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            # Supplying labels allows Hugging Face
            # to calculate CrossEntropyLoss.
            outputs = model(
                **inputs,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            # Probability assigned to class 1: fake
            fake_probabilities = probabilities[
                :,
                1
            ]

            current_batch_size = labels.size(0)

            total_loss += (
                loss.item()
                * current_batch_size
            )

            total_samples += current_batch_size

            all_labels.extend(
                labels.detach().cpu().numpy()
            )

            all_predictions.extend(
                predictions.detach().cpu().numpy()
            )

            all_fake_probabilities.extend(
                fake_probabilities
                .detach()
                .cpu()
                .numpy()
            )


    if total_samples == 0:
        raise RuntimeError(
            "The evaluation DataLoader contains no samples."
        )


    # ========================================================
    # CONVERT TO NUMPY ARRAYS
    # ========================================================

    y_true = np.asarray(
        all_labels,
        dtype=np.int64
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64
    )

    y_score = np.asarray(
        all_fake_probabilities,
        dtype=np.float64
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = confusion.ravel()


    # ========================================================
    # CLASSIFICATION METRICS
    # ========================================================

    average_loss = (
        total_loss / total_samples
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )


    # ========================================================
    # ROC-AUC, PR-AUC, AP, AND EER
    # ========================================================

    if len(np.unique(y_true)) == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_score
        )

        average_precision = (
            average_precision_score(
                y_true,
                y_score
            )
        )

        pr_precision, pr_recall, _ = (
            precision_recall_curve(
                y_true,
                y_score,
                pos_label=1
            )
        )

        pr_auc = auc(
            pr_recall,
            pr_precision
        )

        roc_fpr, roc_tpr, roc_thresholds = (
            roc_curve(
                y_true,
                y_score,
                pos_label=1
            )
        )

        roc_fnr = 1.0 - roc_tpr

        # Approximate equal-error-rate point
        eer_index = np.nanargmin(
            np.abs(
                roc_fpr - roc_fnr
            )
        )

        eer = (
            roc_fpr[eer_index]
            + roc_fnr[eer_index]
        ) / 2.0

        eer_threshold = (
            roc_thresholds[eer_index]
        )

    else:
        roc_auc = np.nan
        pr_auc = np.nan
        average_precision = np.nan
        eer = np.nan
        eer_threshold = np.nan


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[
            "real",
            "fake"
        ],
        digits=4,
        zero_division=0
    )


    # ========================================================
    # RESULT DICTIONARY
    # ========================================================

    results = {
        "test_loss": average_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1_score": f1,
        "mcc": mcc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "average_precision": average_precision,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "number_of_test_images": int(total_samples),
        "confusion_matrix": confusion,
        "classification_report": report
    }


    predictions_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "fake_probability": y_score
    })

    results["predictions"] = predictions_df

    return results

# Wild deepfake

In [20]:
import h5py
import numpy as np

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

with h5py.File(H5_PATH, "r") as h5f:
    # Load image arrays
    train_images = h5f["train_images"][:]
    train_labels = h5f["train_labels"][:]

    val_images = h5f["val_images"][:]
    val_labels = h5f["val_labels"][:]

    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

# Verify dataset sizes
print(f"Total train: {len(train_images)} images")
print(f"Total validation: {len(val_images)} images")
print(f"Total test: {len(test_images)} images")

print(f"Train labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test labels: {len(test_labels)}")

# Verify shapes and data types
print("\nArray information:")
print(f"Train images: {train_images.shape}, dtype={train_images.dtype}")
print(f"Validation images: {val_images.shape}, dtype={val_images.dtype}")
print(f"Test images: {test_images.shape}, dtype={test_images.dtype}")

print(f"Train labels: {train_labels.shape}, dtype={train_labels.dtype}")
print(f"Validation labels: {val_labels.shape}, dtype={val_labels.dtype}")
print(f"Test labels: {test_labels.shape}, dtype={test_labels.dtype}")

# Verify class distributions
print("\nClass distribution:")
print(
    f"Train: Real={np.sum(train_labels == 0)}, "
    f"Fake={np.sum(train_labels == 1)}"
)
print(
    f"Validation: Real={np.sum(val_labels == 0)}, "
    f"Fake={np.sum(val_labels == 1)}"
)
print(
    f"Test: Real={np.sum(test_labels == 0)}, "
    f"Fake={np.sum(test_labels == 1)}"
)

Total train: 36000 images
Total validation: 6000 images
Total test: 18000 images
Train labels: 36000
Validation labels: 6000
Test labels: 18000

Array information:
Train images: (36000, 160, 160, 3), dtype=uint8
Validation images: (6000, 160, 160, 3), dtype=uint8
Test images: (18000, 160, 160, 3), dtype=uint8
Train labels: (36000,), dtype=uint8
Validation labels: (6000,), dtype=uint8
Test labels: (18000,), dtype=uint8

Class distribution:
Train: Real=9000, Fake=27000
Validation: Real=1500, Fake=4500
Test: Real=4500, Fake=13500


In [9]:
# Convert data into PyTorch Dataset
train_dataset = ImageClassificationDataset(train_images, train_labels, processor)
val_dataset = ImageClassificationDataset(val_images, val_labels, processor)
test_dataset = ImageClassificationDataset(test_images, test_labels, processor)

# Dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [10]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=10
)

Epoch 01/10 | Train Loss: 0.3460 | Train Accuracy: 0.8458 | Val Loss: 0.2119 | Val Accuracy: 0.9087
Epoch 02/10 | Train Loss: 0.1547 | Train Accuracy: 0.9404 | Val Loss: 0.2904 | Val Accuracy: 0.8940
Epoch 03/10 | Train Loss: 0.1024 | Train Accuracy: 0.9619 | Val Loss: 0.2598 | Val Accuracy: 0.9212
Epoch 04/10 | Train Loss: 0.0769 | Train Accuracy: 0.9700 | Val Loss: 0.2130 | Val Accuracy: 0.9273
Epoch 05/10 | Train Loss: 0.0623 | Train Accuracy: 0.9772 | Val Loss: 0.2010 | Val Accuracy: 0.9345
Epoch 06/10 | Train Loss: 0.0530 | Train Accuracy: 0.9802 | Val Loss: 0.1890 | Val Accuracy: 0.9373
Epoch 07/10 | Train Loss: 0.0458 | Train Accuracy: 0.9832 | Val Loss: 0.2305 | Val Accuracy: 0.9378
Epoch 08/10 | Train Loss: 0.0421 | Train Accuracy: 0.9847 | Val Loss: 0.2430 | Val Accuracy: 0.9395
Epoch 09/10 | Train Loss: 0.0382 | Train Accuracy: 0.9861 | Val Loss: 0.2880 | Val Accuracy: 0.9258
Epoch 10/10 | Train Loss: 0.0353 | Train Accuracy: 0.9874 | Val Loss: 0.2132 | Val Accuracy: 0.9435


In [11]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [12]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader
)
print("\nSWINV2-TINY TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )

SWINV2-TINY TEST RESULTS
test_loss                     : 0.825046
accuracy                      : 0.8193
balanced_accuracy             : 0.8132
precision                     : 0.9257
recall_sensitivity            : 0.8254
specificity                   : 0.8009
f1_score                      : 0.8727
mcc                           : 0.5763
roc_auc                       : 0.8853
pr_auc                        : 0.9583
average_precision             : 0.9583 
eer                           : 0.1862
eer_threshold                 : 0.400016
false_positive_rate           : 0.1991
false_negative_rate           : 0.1746
true_negatives                : 3604
false_positives               : 896
false_negatives               : 2357
true_positives                : 11143
number_of_test_images         : 18000


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
#print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 12.0%
Time Usage: 261.8 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [15]:
end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 25.0%
Time Usage: 261.9 s
GPU Memory Used: 440.5 MB
Power Consumption: 93W


save the model

In [29]:
# ============================================================
# SAVE SWINV2 MODEL
# ============================================================

SAVE_DIR = (
    r"D:\thesis\results"
    r"\swinv2_tiny_wild_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


model.save_pretrained(
    SAVE_DIR
)

processor.save_pretrained(
    SAVE_DIR
)


torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,



        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },

    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)


print("SwinV2 model saved to:")
print(SAVE_DIR)

SwinV2 model saved to:
D:\thesis\results\swinv2_tiny_wild_160


load the model

In [16]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_39424\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [18]:
# ============================================================
# PYTORCH SWINV2 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # Supports a dataset returning one dictionary:
        # {"pixel_values": ..., "labels": ...}
        if isinstance(batch, dict):

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        # Supports a dataset returning:
        # inputs_dictionary, labels
        else:

            inputs, labels = batch

            pixel_values = inputs[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        outputs = model(
            pixel_values=pixel_values
        )

        logits = outputs.logits

        # Access one value to ensure output creation
        _ = logits[-1, 0]


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting SwinV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            if isinstance(batch, dict):

                pixel_values = batch[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            else:

                inputs, labels = batch

                pixel_values = inputs[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            outputs = model(
                pixel_values=pixel_values
            )

            logits = outputs.logits

            last_logits = logits

            processed_images += (
                pixel_values.size(0)
            )


    # CUDA execution is asynchronous
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del outputs


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual SwinV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 18000
Test batches: 1125

Performing warm-up using 3 batches...
Warm-up completed.

Starting SwinV2 resource run 1/5
Run 1: 222.90 seconds | 12.3832 ms/image | 80.75 images/s

Starting SwinV2 resource run 2/5
Run 2: 232.02 seconds | 12.8899 ms/image | 77.58 images/s

Starting SwinV2 resource run 3/5
Run 3: 243.75 seconds | 13.5417 ms/image | 73.85 images/s

Starting SwinV2 resource run 4/5
Run 4: 243.65 seconds | 13.5359 ms/image | 73.88 images/s

Starting SwinV2 resource run 5/5
Run 5: 239.98 seconds | 13.3321 ms/image | 75.01 images/s

Individual SwinV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,222.897182,12.383177,80.754722,3.140018,8.434375,5334.281250,5323.452991,5335.789062,0.0,1.507812,...,531.194766,532.000000,39.888777,99,36.231356,102.759,2.247058,1,18000,-0.622375
1,232.018459,12.889914,77.580034,3.139321,11.946875,5324.542969,5317.696667,5332.000000,0.0,7.457031,...,531.226306,532.000000,38.046422,100,34.034108,112.206,2.198482,2,18000,-0.622375
2,243.749837,13.541658,73.846203,3.140430,8.434375,5316.601562,5300.104711,5328.710938,0.0,12.109375,...,586.999147,603.429688,36.331494,99,33.432426,101.650,2.267976,3,18000,-0.622375
3,243.645877,13.535882,73.877712,3.135097,10.387500,5290.261719,5285.955747,5302.558594,0.0,12.296875,...,531.274705,532.000000,36.799184,87,30.686217,91.251,2.077999,4,18000,-0.622375
4,239.977771,13.332098,75.006947,3.109967,8.512500,5278.917969,5278.616860,5291.140625,0.0,12.222656,...,531.258916,532.000000,37.064845,100,31.448771,92.993,2.100164,5,18000,-0.622375


In [19]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("SwinV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


SwinV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,236.457825,8.955872,225.337641,247.578009
1,latency_ms_per_image,13.136546,0.497548,12.518758,13.754334
2,throughput_images_per_s,76.213124,2.958288,72.539924,79.886324
3,average_cpu_percent,3.132967,0.013032,3.116785,3.149148
4,peak_cpu_percent,9.543125,1.582073,7.578721,11.507529
5,average_ram_mb,5301.165395,19.435243,5277.033353,5325.297437
6,peak_ram_mb,5318.039844,19.918773,5293.307421,5342.772267
7,average_incremental_ram_mb,0.000000,0.000000,0.000000,0.000000
8,peak_incremental_ram_mb,9.118750,4.726686,3.249795,14.987705
9,average_gpu_memory_mb,2988.694674,33.883906,2946.622249,3030.767100



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 13.137 ± 0.498 (95% CI: 12.519–13.754)
peak_ram_mb: 5318.040 ± 19.919 (95% CI: 5293.307–5342.772)
peak_gpu_memory_mb: 2992.590 ± 36.088 (95% CI: 2947.781–3037.398)
average_gpu_utilization_percent: 37.626 ± 1.412 (95% CI: 35.873–39.379)
average_gpu_power_w: 33.167 ± 2.198 (95% CI: 30.438–35.895)


genralization

In [21]:
print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (SWINV2):")
test_dataset = ImageClassificationDataset(test_celeb, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of wild deepfake dataset on Celeb-DF(V2) (SWINV2):
test_loss                     : 0.805100
accuracy                      : 0.774702
balanced_accuracy             : 0.639767
precision                     : 0.935255
recall_sensitivity            : 0.806679
specificity                   : 0.472855
f1_score                      : 0.866222
mcc                           : 0.198564
roc_auc                       : 0.727252
pr_auc                        : 0.958911
average_precision             : 0.958913
eer                           : 0.331269
eer_threshold                 : 0.929526
false_positive_rate           : 0.527145
false_negative_rate           : 0.193321
true_negatives                : 270
false_positives               : 301
false_negatives               : 1042
true_positives                : 4348
number_of_test_images         : 5961


In [23]:
#dfc on wilddeepfake
print("\nTest results of wild deepfake dataset on DFC (SWINV2):")
test_dataset = ImageClassificationDataset(test_hog, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )



Test results of wild deepfake dataset on DFC (SWINV2):
test_loss                     : 2.167923
accuracy                      : 0.526333
balanced_accuracy             : 0.526333
precision                     : 0.545455
recall_sensitivity            : 0.316000
specificity                   : 0.736667
f1_score                      : 0.400169
mcc                           : 0.058053
roc_auc                       : 0.533985
pr_auc                        : 0.528971
average_precision             : 0.529570
eer                           : 0.483333
eer_threshold                 : 0.057846
false_positive_rate           : 0.263333
false_negative_rate           : 0.684000
true_negatives                : 1105
false_positives               : 395
false_negatives               : 1026
true_positives                : 474
number_of_test_images         : 3000


In [25]:
print("\nTest results of wild deepfake dataset on FF++ (SWINV2):")
#ff++ on wilddeepfake
test_dataset = ImageClassificationDataset(test_ff, test_ff_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of wild deepfake dataset on FF++ (SWINV2):
test_loss                     : 1.955783
accuracy                      : 0.530101
balanced_accuracy             : 0.535298
precision                     : 0.449231
recall_sensitivity            : 0.566440
specificity                   : 0.504155
f1_score                      : 0.501073
mcc                           : 0.069695
roc_auc                       : 0.542259
pr_auc                        : 0.466339
average_precision             : 0.466905
eer                           : 0.458613
eer_threshold                 : 0.590140
false_positive_rate           : 0.495845
false_negative_rate           : 0.433560
true_negatives                : 728
false_positives               : 716
false_negatives               : 447
true_positives                : 584
number_of_test_images         : 2475


# Celeb

In [22]:
import os
import cv2
import numpy as np

SAVE_ROOT = r'D:\thesis\celeb_processed'

def load_split(split_name, class_name):
    """Reload saved frames, grouped by video."""
    base = os.path.join(SAVE_ROOT, split_name, class_name)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        vid_dir = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(vid_dir, f))
                  for f in sorted(os.listdir(vid_dir))]
        if frames:
            nested.append(frames)
            ids.append(vid_id)
    return nested, ids

# Reload ALL six splits
print("Loading frames...")
real_train_final,  real_train_ids  = load_split('train', 'real')
synth_train_final, synth_train_ids = load_split('train', 'fake')
real_val_final,    real_val_ids    = load_split('val',   'real')
synth_val_final,   synth_val_ids   = load_split('val',   'fake')
real_test_final,   real_test_ids   = load_split('test',  'real')
synth_test_final,  synth_test_ids  = load_split('test',  'fake')

print("✅ All frames reloaded")
print("Train -> real videos:", len(real_train_final), " fake videos:", len(synth_train_final))
print("Val   -> real videos:", len(real_val_final),   " fake videos:", len(synth_val_final))
print("Test  -> real videos:", len(real_test_final),  " fake videos:", len(synth_test_final))
print("Example frame shape:", np.shape(real_train_final[0][0]))  # expect (160,160,3)
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Flatten video-grouped frames into one image array and create labels.

    real_videos: list of videos, where each video is a list of frames
    fake_videos: list of videos, where each video is a list of frames

    Returns
    -------
    images : NumPy array with shape (N, 160, 160, 3)
    labels : NumPy array with shape (N,)
             0 = real, 1 = fake
    """

    # Flatten frames from all real videos
    real_frames = [
        frame
        for video_frames in real_videos
        for frame in video_frames
        if frame is not None
    ]

    # Flatten frames from all fake videos
    fake_frames = [
        frame
        for video_frames in fake_videos
        for frame in video_frames
        if frame is not None
    ]

    if len(real_frames) == 0:
        raise ValueError("No real frames were found.")

    if len(fake_frames) == 0:
        raise ValueError("No fake frames were found.")

    # Convert to NumPy arrays
    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    # Combine images
    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    # Create labels
    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training set
train_celeb, train_labels = combine_split(
    real_train_final,
    synth_train_final
)

# Validation set
val_celeb, val_labels = combine_split(
    real_val_final,
    synth_val_final
)

# Testing set
test_celeb, test_labels = combine_split(
    real_test_final,
    synth_test_final
)
print("\nTRAIN")
print("Images:", train_celeb.shape)
print("Labels:", train_labels.shape)
print("Real:", np.sum(train_labels == 0))
print("Fake:", np.sum(train_labels == 1))

print("\nVALIDATION")
print("Images:", val_celeb.shape)
print("Labels:", val_labels.shape)
print("Real:", np.sum(val_labels == 0))
print("Fake:", np.sum(val_labels == 1))

print("\nTEST")
print("Images:", test_celeb.shape)
print("Labels:", test_labels.shape)
print("Real:", np.sum(test_labels == 0))
print("Fake:", np.sum(test_labels == 1))

print("\nData types")
print("Train images:", train_celeb.dtype)
print("Train labels:", train_labels.dtype)

Loading frames...
✅ All frames reloaded
Train -> real videos: 354  fake videos: 3383
Val   -> real videos: 59  fake videos: 563
Test  -> real videos: 177  fake videos: 1693
Example frame shape: (160, 160, 3)

TRAIN
Images: (11899, 160, 160, 3)
Labels: (11899,)
Real: 1142
Fake: 10757

VALIDATION
Images: (1969, 160, 160, 3)
Labels: (1969,)
Real: 182
Fake: 1787

TEST
Images: (5961, 160, 160, 3)
Labels: (5961,)
Real: 571
Fake: 5390

Data types
Train images: uint8
Train labels: uint8


In [9]:
# Convert data into PyTorch Dataset
train_dataset = ImageClassificationDataset(train_celeb, train_labels, processor)
val_dataset = ImageClassificationDataset(val_celeb, val_labels, processor)
test_dataset = ImageClassificationDataset(test_celeb, test_labels, processor)

# Dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [10]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=10
)

Epoch 01/10 | Train Loss: 0.2981 | Train Accuracy: 0.9031 | Val Loss: 0.2376 | Val Accuracy: 0.9076
Epoch 02/10 | Train Loss: 0.2596 | Train Accuracy: 0.9030 | Val Loss: 0.2199 | Val Accuracy: 0.9096
Epoch 03/10 | Train Loss: 0.2021 | Train Accuracy: 0.9218 | Val Loss: 0.1434 | Val Accuracy: 0.9431
Epoch 04/10 | Train Loss: 0.1451 | Train Accuracy: 0.9467 | Val Loss: 0.1044 | Val Accuracy: 0.9650
Epoch 05/10 | Train Loss: 0.0953 | Train Accuracy: 0.9645 | Val Loss: 0.0945 | Val Accuracy: 0.9650
Epoch 06/10 | Train Loss: 0.0753 | Train Accuracy: 0.9733 | Val Loss: 0.1450 | Val Accuracy: 0.9518
Epoch 07/10 | Train Loss: 0.0566 | Train Accuracy: 0.9797 | Val Loss: 0.0813 | Val Accuracy: 0.9726
Epoch 08/10 | Train Loss: 0.0403 | Train Accuracy: 0.9859 | Val Loss: 0.1226 | Val Accuracy: 0.9695
Epoch 09/10 | Train Loss: 0.0539 | Train Accuracy: 0.9812 | Val Loss: 0.1024 | Val Accuracy: 0.9680
Epoch 10/10 | Train Loss: 0.0423 | Train Accuracy: 0.9855 | Val Loss: 0.0984 | Val Accuracy: 0.9721


In [11]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [12]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader
)
print("\nSWINV2-TINY TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


SWINV2-TINY TEST RESULTS
test_loss                     : 0.107990
accuracy                      : 0.974333
balanced_accuracy             : 0.879333
precision                     : 0.975313
recall_sensitivity            : 0.996846
specificity                   : 0.761821
f1_score                      : 0.985962
mcc                           : 0.843448
roc_auc                       : 0.990014
pr_auc                        : 0.998829
average_precision             : 0.997704
eer                           : 0.045123
eer_threshold                 : 0.983558
false_positive_rate           : 0.238179
false_negative_rate           : 0.003154
true_negatives                : 435
false_positives               : 136
false_negatives               : 17
true_positives                : 5373
number_of_test_images         : 5961
 


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 17.9%
Time Usage: 179.1 s
GPU Memory Used: 0.0 MB
Power Consumption: 137W


In [15]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 16.1%
Time Usage: 181.3 s
GPU Memory Used: 440.5 MB
Power Consumption: 93W


save the model

In [16]:
# ============================================================
# SAVE SWINV2 MODEL
# ============================================================

SAVE_DIR = (
    r"D:\thesis\results"
    r"\swinv2_tiny_celeb_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


model.save_pretrained(
    SAVE_DIR
)

processor.save_pretrained(
    SAVE_DIR
)


torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },

    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)


print("SwinV2 model saved to:")
print(SAVE_DIR)

SwinV2 model saved to:
D:\thesis\results\swinv2_tiny_celeb_160


In [17]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_9940\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [18]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


In [20]:
# ============================================================
# PYTORCH SWINV2 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # Supports a dataset returning one dictionary:
        # {"pixel_values": ..., "labels": ...}
        if isinstance(batch, dict):

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        # Supports a dataset returning:
        # inputs_dictionary, labels
        else:

            inputs, labels = batch

            pixel_values = inputs[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        outputs = model(
            pixel_values=pixel_values
        )

        logits = outputs.logits

        # Access one value to ensure output creation
        _ = logits[-1, 0]


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting SwinV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            if isinstance(batch, dict):

                pixel_values = batch[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            else:

                inputs, labels = batch

                pixel_values = inputs[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            outputs = model(
                pixel_values=pixel_values
            )

            logits = outputs.logits

            last_logits = logits

            processed_images += (
                pixel_values.size(0)
            )


    # CUDA execution is asynchronous
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del outputs


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual SwinV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 5961
Test batches: 373

Performing warm-up using 3 batches...
Warm-up completed.

Starting SwinV2 resource run 1/5
Run 1: 77.15 seconds | 12.9430 ms/image | 77.26 images/s

Starting SwinV2 resource run 2/5
Run 2: 78.52 seconds | 13.1731 ms/image | 75.91 images/s

Starting SwinV2 resource run 3/5
Run 3: 80.40 seconds | 13.4875 ms/image | 74.14 images/s

Starting SwinV2 resource run 4/5
Run 4: 79.92 seconds | 13.4076 ms/image | 74.58 images/s

Starting SwinV2 resource run 5/5
Run 5: 77.25 seconds | 12.9589 ms/image | 77.17 images/s

Individual SwinV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,77.153239,12.943003,77.261824,3.110651,6.825000,2379.460938,2368.201753,2379.972656,0.000000,0.511719,...,529.724040,532.0,40.600284,99,33.137529,81.328,0.711351,1,5961,-5.203214
1,78.524979,13.173122,75.912150,3.129738,7.103125,2363.496094,2369.273999,2381.046875,5.777906,17.550781,...,529.743300,532.0,38.809591,100,33.566700,93.681,0.733798,2,5961,-5.203214
2,80.398778,13.487465,74.142918,3.097721,8.062500,2363.937500,2369.205407,2381.238281,5.267907,17.300781,...,529.811218,532.0,37.839945,100,33.764860,91.708,0.755989,3,5961,-5.203214
3,79.922913,13.407635,74.584369,3.101967,8.512500,2358.140625,2363.936704,2374.832031,5.796079,16.691406,...,529.796143,532.0,37.292011,99,30.411008,78.591,0.676626,4,5961,-5.203214
4,77.248010,12.958901,77.167037,3.057726,7.546875,2358.488281,2364.167997,2376.035156,5.679715,17.546875,...,529.724040,532.0,37.381223,100,28.775390,67.747,0.618778,5,5961,-5.203214


In [21]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("SwinV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


SwinV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,78.649584,1.491646,76.797461,80.501707
1,latency_ms_per_image,13.194025,0.250234,12.883318,13.504732
2,throughput_images_per_s,75.813659,1.435361,74.031423,77.595896
3,average_cpu_percent,3.099561,0.026424,3.066750,3.132371
4,peak_cpu_percent,7.610000,0.688548,6.755054,8.464946
5,average_ram_mb,2366.957172,2.686725,2363.621162,2370.293181
6,peak_ram_mb,2378.625000,2.983488,2374.920511,2382.329489
7,average_incremental_ram_mb,4.504321,2.527062,1.366559,7.642084
8,peak_incremental_ram_mb,13.920312,7.503819,4.603091,23.237534
9,average_gpu_memory_mb,3246.866779,5.346723,3240.227946,3253.505613



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 13.194 ± 0.250 (95% CI: 12.883–13.505)
peak_ram_mb: 2378.625 ± 2.983 (95% CI: 2374.921–2382.329)
peak_gpu_memory_mb: 3249.107 ± 5.367 (95% CI: 3242.444–3255.770)
average_gpu_utilization_percent: 38.385 ± 1.377 (95% CI: 36.675–40.095)
average_gpu_power_w: 31.931 ± 2.223 (95% CI: 29.171–34.691)


#genralization

In [23]:
#wild deepfake on celeb
print("\nTest results of Celeb-DF(V2) on wild deepfake dataset (SWINV2):")
test_dataset = ImageClassificationDataset(test_images, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )



Test results of Celeb-DF(V2) on wild deepfake dataset (SWINV2):
test_loss                     : 1.762938
accuracy                      : 0.546000
balanced_accuracy             : 0.564741
precision                     : 0.799057
recall_sensitivity            : 0.527259
specificity                   : 0.602222
f1_score                      : 0.635309
mcc                           : 0.112140
roc_auc                       : 0.588251
pr_auc                        : 0.811872
average_precision             : 0.811928
eer                           : 0.435148
eer_threshold                 : 0.375871
false_positive_rate           : 0.397778
false_negative_rate           : 0.472741
true_negatives                : 2710
false_positives               : 1790
false_negatives               : 6382
true_positives                : 7118
number_of_test_images         : 18000


In [25]:
#DFC on celeb
print("\nTest results of Celeb-DF(V2) on DFC dataset (SWINV2):")
#ff++ on wilddeepfake
test_dataset = ImageClassificationDataset(test_hog, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of Celeb-DF(V2) on DFC dataset (SWINV2):
test_loss                     : 2.737444
accuracy                      : 0.432667
balanced_accuracy             : 0.432667
precision                     : 0.458368
recall_sensitivity            : 0.741333
specificity                   : 0.124000
f1_score                      : 0.566480
mcc                           : -0.171179
roc_auc                       : 0.409529
pr_auc                        : 0.448665
average_precision             : 0.449550
eer                           : 0.557000
eer_threshold                 : 0.986693
false_positive_rate           : 0.876000
false_negative_rate           : 0.258667
true_negatives                : 186
false_positives               : 1314
false_negatives               : 388
true_positives                : 1112
number_of_test_images         : 3000


In [27]:
print("\nTest results of wild deepfake dataset on FF++ (SWINV2):")
#ff++ on wilddeepfake
test_dataset = ImageClassificationDataset(test_ff, test_ff_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of wild deepfake dataset on FF++ (SWINV2):
test_loss                     : 1.444258
accuracy                      : 0.618990
balanced_accuracy             : 0.661825
precision                     : 0.524363
recall_sensitivity            : 0.918526
specificity                   : 0.405125
f1_score                      : 0.667607
mcc                           : 0.359267
roc_auc                       : 0.846245
pr_auc                        : 0.826746
average_precision             : 0.826830
eer                           : 0.221861
eer_threshold                 : 0.986536
false_positive_rate           : 0.594875
false_negative_rate           : 0.081474
true_negatives                : 585
false_positives               : 859
false_negatives               : 84
true_positives                : 947
number_of_test_images         : 2475


# DFC

In [24]:
import h5py
import numpy as np
# Open the HDF5 file in read mode
with h5py.File('D://thesis//dataset//deepfake dataset//resized_images.h5', 'r') as h5f:
    # Access each dataset
    celeb = np.array(h5f['celeb'])
    ffhq = np.array(h5f['ffhq'])
    gdwct = np.array(h5f['gdwct'])
    attgan = np.array(h5f['attgan'])
    stargan = np.array(h5f['stargan'])
    stylegan2 = np.array(h5f['stylegan2'])
    stylegan = np.array(h5f['stylegan'])

# Now, 'celeb', 'ffhq', etc., are NumPy arrays containing your datasets
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"ffhq shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"ffhq shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"ffhq shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"ffhq shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"ffhq shape: {stylegan.shape}, dtype: {stylegan.dtype}")
# Repeat for other datasets as needed
import cv2
# Function to resize images from (224, 224) to (160, 160)
def resize_images(image_array, target_size=(160, 160)):
    resized_images = np.array([cv2.resize(img, target_size) for img in image_array])
    return resized_images

celeb = resize_images(celeb, target_size=(160, 160))
ffhq = resize_images(ffhq, target_size=(160, 160))
gdwct = resize_images(gdwct, target_size=(160, 160))
attgan = resize_images(attgan, target_size=(160, 160))
stargan = resize_images(stargan, target_size=(160, 160))
stylegan = resize_images(stylegan, target_size=(160, 160))
stylegan2 = resize_images(stylegan2, target_size=(160, 160))
import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(celeb)), 2500)  # Get 2500 random indices
celeb = celeb[random_indices]  # Select the random subse

import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(ffhq)), 2500)  # Get 2500 random indices
ffhq = ffhq[random_indices]  # Select the random subse
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"gdwct shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"attagan shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"stargan shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"stylegan2 shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"stylegan shape: {stylegan.shape}, dtype: {stylegan.dtype}")
import random
import numpy as np

def split_data(data, train_ratio=0.7):
    """
    Splits data into training and testing sets based on the specified ratio.

    Parameters:
        data (list or np.array): The dataset to split.
        train_ratio (float): The ratio of the data to include in the training set.

    Returns:
        tuple: Two datasets - train and test.
    """
    # Shuffle the data
    random.shuffle(data)

    # Calculate the split index
    split_index = int(len(data) * train_ratio)

    # Split the data
    train_data = data[:split_index]
    test_data = data[split_index:]

    return train_data, test_data

# Split `celeb` into 70% train and 30% test
celeb_train_hog, celeb_test_hog = split_data(celeb, train_ratio=0.7)

# Split `ffhq` into 70% train and 30% test
ffhq_train_hog, ffhq_test_hog = split_data(ffhq, train_ratio=0.7)

# Split `attgan` into 70% train and 30% test
attgan_train_hog, attgan_test_hog = split_data(attgan, train_ratio=0.7)

# Split `stargan` into 70% train and 30% test
stargan_train_hog, stargan_test_hog = split_data(stargan, train_ratio=0.7)

# Split `gdwct` into 70% train and 30% test
gdwct_train_hog, gdwct_test_hog = split_data(gdwct, train_ratio=0.7)

# Split `stylegan2` into 70% train and 30% test_hog
stylegan2_train_hog, stylegan2_test_hog = split_data(stylegan2, train_ratio=0.7)

# Split `stylegan` into 70% train and 30% test_hog
stylegan_train_hog, stylegan_test_hog = split_data(stylegan, train_ratio=0.7)

# Convert to NumPy arrays if needed
celeb_train_hog, celeb_test_hog = np.array(celeb_train_hog), np.array(celeb_test_hog)
ffhq_train_hog, ffhq_test_hog = np.array(ffhq_train_hog), np.array(ffhq_test_hog)
attgan_train_hog, attgan_test_hog = np.array(attgan_train_hog), np.array(attgan_test_hog)
stargan_train_hog, stargan_test_hog = np.array(stargan_train_hog), np.array(stargan_test_hog)
gdwct_train_hog, gdwct_test_hog = np.array(gdwct_train_hog), np.array(gdwct_test_hog)
stylegan2_train_hog, stylegan2_test_hog = np.array(stylegan2_train_hog), np.array(stylegan2_test_hog)
stylegan_train_hog, stylegan_test_hog = np.array(stylegan_train_hog), np.array(stylegan_test_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_test: {len(celeb_test_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_test: {len(ffhq_test_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_test: {len(attgan_test_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_test: {len(stargan_test_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_test: {len(gdwct_test_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_test: {len(stylegan2_test_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_test: {len(stylegan_test_hog)} images")

########################################################################################################################################
#######################################divide into 60,10 train and val
#########################################################################################################################################
def extract_validation(train_data):
    """
    Extract every 10th sample from the training data and store it in a validation set.

    Parameters:
        train_data (list or np.array): The training dataset.

    Returns:
        tuple: Updated training dataset and validation dataset.
    """
    # Select every 10th sample for the validation set
    validation_data = train_data[::10]

    # Remove the selected samples from the training dataset
    updated_train_data = [train_data[i] for i in range(len(train_data)) if i % 10 != 0]

    return np.array(updated_train_data), np.array(validation_data)


# Perform the operation for each dataset
celeb_train_hog, celeb_val_hog = extract_validation(celeb_train_hog)
ffhq_train_hog, ffhq_val_hog = extract_validation(ffhq_train_hog)
attgan_train_hog, attgan_val_hog = extract_validation(attgan_train_hog)
stargan_train_hog, stargan_val_hog = extract_validation(stargan_train_hog)
gdwct_train_hog, gdwct_val_hog = extract_validation(gdwct_train_hog)
stylegan2_train_hog, stylegan2_val_hog = extract_validation(stylegan2_train_hog)
stylegan_train_hog, stylegan_val_hog = extract_validation(stylegan_train_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_val: {len(celeb_val_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_val: {len(ffhq_val_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_val: {len(attgan_val_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_val: {len(stargan_val_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_val: {len(gdwct_val_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_val: {len(stylegan2_val_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_val: {len(stylegan_val_hog)} images")
############################################################################################################################################################
#################################################concatenate the labels 0,1 real and fake
#############################################################################################################################################################


celeb_train_labels = np.zeros(len(celeb_train_hog), dtype=int)
ffhq_train_labels = np.zeros(len(ffhq_train_hog), dtype=int)
atta_train_labels = np.ones(len(attgan_train_hog), dtype=int)
star_train_labels = np.ones(len(stargan_train_hog), dtype=int)
gdwct_train_labels = np.ones(len(gdwct_train_hog), dtype=int)
stylegan2_train_labels = np.ones(len(stylegan2_train_hog), dtype=int)
stylegan_train_labels = np.ones(len(stylegan_train_hog), dtype=int)

# Concatenate all training datasets into a single `train` variable
train_hog = np.concatenate([celeb_train_hog, ffhq_train_hog, attgan_train_hog, stargan_train_hog, gdwct_train_hog, stylegan2_train_hog, stylegan_train_hog], axis=0)
train_labels=np.concatenate([celeb_train_labels, ffhq_train_labels, atta_train_labels, star_train_labels, gdwct_train_labels, stylegan2_train_labels,
                              stylegan_train_labels], axis=0)




celeb_test_labels = np.zeros(len(celeb_test_hog), dtype=int)
ffhq_test_labels = np.zeros(len(ffhq_test_hog), dtype=int)
atta_test_labels = np.ones(len(attgan_test_hog), dtype=int)
star_test_labels = np.ones(len(stargan_test_hog), dtype=int)
gdwct_test_labels = np.ones(len(gdwct_test_hog), dtype=int)
stylegan2_test_labels = np.ones(len(stylegan2_test_hog), dtype=int)
stylegan_test_labels = np.ones(len(stylegan_test_hog), dtype=int)

# Concatenate all testing datasets into a single `test` variable
test_hog = np.concatenate([celeb_test_hog, ffhq_test_hog, attgan_test_hog, stargan_test_hog, gdwct_test_hog, stylegan2_test_hog, stylegan_test_hog], axis=0)
test_labels = np.concatenate([celeb_test_labels, ffhq_test_labels, atta_test_labels, star_test_labels, gdwct_test_labels, stylegan2_test_labels,
                        stylegan_test_labels], axis=0)




celeb_val_labels = np.zeros(len(celeb_val_hog), dtype=int)
ffhq_val_labels = np.zeros(len(ffhq_val_hog), dtype=int)
atta_val_labels = np.ones(len(attgan_val_hog), dtype=int)
star_val_labels = np.ones(len(stargan_val_hog), dtype=int)
gdwct_val_labels = np.ones(len(gdwct_val_hog), dtype=int)
stylegan2_val_labels = np.ones(len(stylegan2_val_hog), dtype=int)
stylegan_val_labels = np.ones(len(stylegan_val_hog), dtype=int)

# Concatenate all validation datasets into a single `val` variable
val_hog = np.concatenate([celeb_val_hog, ffhq_val_hog, attgan_val_hog, stargan_val_hog, gdwct_val_hog, stylegan2_val_hog, stylegan_val_hog], axis=0)
val_labels = np.concatenate([celeb_val_labels, ffhq_val_labels, atta_val_labels, star_val_labels, gdwct_val_labels, stylegan2_val_labels,
                       stylegan_val_labels], axis=0)

# Print the results for verification
print(f"Total train: {len(train_hog)} images")
print(f"Total test: {len(test_hog)} images")
print(f"Total val: {len(val_hog)} images")


# Print results for verification
print(f"Train Labels: {len(train_labels)} ")
print(f"Test Labels: {len(test_labels)} ")
print(f"Val Labels: {len(val_labels)} ")



celeb shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
celeb shape: (2500, 160, 160, 3), dtype: uint8
ffhq shape: (2500, 160, 160, 3), dtype: uint8
gdwct shape: (1000, 160, 160, 3), dtype: uint8
attagan shape: (1000, 160, 160, 3), dtype: uint8
stargan shape: (1000, 160, 160, 3), dtype: uint8
stylegan2 shape: (1000, 160, 160, 3), dtype: uint8
stylegan shape: (1000, 160, 160, 3), dtype: uint8
celeb_train: 1750 images, celeb_test: 750 images
ffhq_train: 1750 images, ffhq_test: 750 images
attgan_train: 700 images, attgan_test: 300 images
stargan_train: 700 images, stargan_test: 300 images
gdwct_train: 700 images, gdwct_test: 300 images
stylegan2_train: 700 images, stylegan2_test: 300 images
stylegan_train: 700 images, stylegan

In [10]:
# Convert data into PyTorch Dataset
train_dataset = ImageClassificationDataset(train_hog, train_labels, processor)
val_dataset = ImageClassificationDataset(val_hog, val_labels, processor)
test_dataset = ImageClassificationDataset(test_hog, test_labels, processor)

# Dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [16]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=10
)

Epoch 01/10 | Train Loss: 0.2134 | Train Accuracy: 0.9032 | Val Loss: 0.0841 | Val Accuracy: 0.9657
Epoch 02/10 | Train Loss: 0.0724 | Train Accuracy: 0.9779 | Val Loss: 0.0305 | Val Accuracy: 0.9886
Epoch 03/10 | Train Loss: 0.0477 | Train Accuracy: 0.9830 | Val Loss: 0.0233 | Val Accuracy: 0.9886
Epoch 04/10 | Train Loss: 0.0377 | Train Accuracy: 0.9867 | Val Loss: 0.0280 | Val Accuracy: 0.9886
Epoch 05/10 | Train Loss: 0.0296 | Train Accuracy: 0.9910 | Val Loss: 0.0240 | Val Accuracy: 0.9900
Epoch 06/10 | Train Loss: 0.0249 | Train Accuracy: 0.9927 | Val Loss: 0.0276 | Val Accuracy: 0.9914
Epoch 07/10 | Train Loss: 0.0172 | Train Accuracy: 0.9954 | Val Loss: 0.0151 | Val Accuracy: 0.9957
Epoch 08/10 | Train Loss: 0.0278 | Train Accuracy: 0.9906 | Val Loss: 0.0221 | Val Accuracy: 0.9886
Epoch 09/10 | Train Loss: 0.0251 | Train Accuracy: 0.9916 | Val Loss: 0.0472 | Val Accuracy: 0.9871
Epoch 10/10 | Train Loss: 0.0265 | Train Accuracy: 0.9914 | Val Loss: 0.0486 | Val Accuracy: 0.9900


In [17]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [18]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [19]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader
)
print("\nSWINV2-TINY TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


SWINV2-TINY TEST RESULTS
test_loss                     : 0.045423
accuracy                      : 0.990667
balanced_accuracy             : 0.990667
precision                     : 0.988712
recall_sensitivity            : 0.992667
specificity                   : 0.988667
f1_score                      : 0.990685
mcc                           : 0.981341
roc_auc                       : 0.997830
pr_auc                        : 0.995016
average_precision             : 0.994512
eer                           : 0.009000
eer_threshold                 : 0.661510
false_positive_rate           : 0.011333
false_negative_rate           : 0.007333
true_negatives                : 1483
false_positives               : 17
false_negatives               : 11
true_positives                : 1489
number_of_test_images         : 3000


In [20]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 21.0%
Time Usage: 43.7 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [21]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 10.0%
Time Usage: 45.8 s
GPU Memory Used: 440.5 MB
Power Consumption: 93W


save the model

CrossViT checkpoint saved:
D:\thesis\results\crossvit_15_160_dfc\crossvit_15_checkpoint_dfc.pt


#load the model

In [22]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


In [23]:
# ============================================================
# PYTORCH SWINV2 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # Supports a dataset returning one dictionary:
        # {"pixel_values": ..., "labels": ...}
        if isinstance(batch, dict):

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        # Supports a dataset returning:
        # inputs_dictionary, labels
        else:

            inputs, labels = batch

            pixel_values = inputs[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        outputs = model(
            pixel_values=pixel_values
        )

        logits = outputs.logits

        # Access one value to ensure output creation
        _ = logits[-1, 0]


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting SwinV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            if isinstance(batch, dict):

                pixel_values = batch[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            else:

                inputs, labels = batch

                pixel_values = inputs[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            outputs = model(
                pixel_values=pixel_values
            )

            logits = outputs.logits

            last_logits = logits

            processed_images += (
                pixel_values.size(0)
            )


    # CUDA execution is asynchronous
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del outputs


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual SwinV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 3000
Test batches: 188

Performing warm-up using 3 batches...
Warm-up completed.

Starting SwinV2 resource run 1/5
Run 1: 40.23 seconds | 13.4096 ms/image | 74.57 images/s

Starting SwinV2 resource run 2/5
Run 2: 40.94 seconds | 13.6472 ms/image | 73.28 images/s

Starting SwinV2 resource run 3/5
Run 3: 38.42 seconds | 12.8063 ms/image | 78.09 images/s

Starting SwinV2 resource run 4/5
Run 4: 37.64 seconds | 12.5455 ms/image | 79.71 images/s

Starting SwinV2 resource run 5/5
Run 5: 38.57 seconds | 12.8583 ms/image | 77.77 images/s

Individual SwinV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,40.228727,13.409576,74.573575,3.072539,6.718750,6836.160156,6824.750160,6836.359375,0.000000,0.199219,...,527.640327,532.0,39.844687,99,32.668744,96.019,0.366934,1,3000,-3.285041
1,40.941634,13.647211,73.275044,3.085057,6.718750,6818.687500,6824.860007,6836.792969,6.172507,18.105469,...,527.692722,532.0,37.865229,99,33.078065,84.778,0.377298,2,3000,-3.285041
2,38.419018,12.806339,78.086328,3.034437,7.990625,6818.703125,6824.897236,6834.910156,6.194111,16.207031,...,527.441595,532.0,39.031339,100,34.803897,79.636,0.372571,3,3000,-3.285041
3,37.636410,12.545470,79.710048,3.095842,8.062500,6818.710938,6824.922215,6829.839844,6.211277,11.128906,...,527.368116,532.0,38.423188,100,29.462780,69.786,0.309215,4,3000,-3.285041
4,38.574964,12.858321,77.770650,3.125909,9.406250,6818.722656,6824.921842,6834.058594,6.199186,15.335938,...,527.485876,532.0,38.279661,100,29.690514,68.541,0.319361,5,3000,-3.285041


In [24]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


Cross-ViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,39.160150,1.371955,37.456643,40.863658
1,latency_ms_per_image,13.053383,0.457318,12.485548,13.621219
2,throughput_images_per_s,76.683129,2.663605,73.375826,79.990431
3,average_cpu_percent,3.082757,0.033458,3.041213,3.124301
4,peak_cpu_percent,7.779375,1.120419,6.388190,9.170560
5,average_ram_mb,6824.870292,0.071793,6824.781149,6824.959434
6,peak_ram_mb,6834.392187,2.772047,6830.950236,6837.834139
7,average_incremental_ram_mb,4.955416,2.770197,1.515762,8.395070
8,peak_incremental_ram_mb,12.195312,7.175145,3.286193,21.104432
9,average_gpu_memory_mb,3254.295259,0.136499,3254.125772,3254.464745



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 13.053 ± 0.457 (95% CI: 12.486–13.621)
peak_ram_mb: 6834.392 ± 2.772 (95% CI: 6830.950–6837.834)
peak_gpu_memory_mb: 3258.770 ± 0.000 (95% CI: 3258.770–3258.770)
average_gpu_utilization_percent: 38.689 ± 0.770 (95% CI: 37.733–39.645)
average_gpu_power_w: 31.941 ± 2.304 (95% CI: 29.081–34.801)


#genralization

In [26]:
#wild deepfake on dfc
print("\nTest results of DFC on wild deepfake dataset (SwinV2):")
test_dataset = ImageClassificationDataset(test_images, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )



Test results of DFC on wild deepfake dataset (SwinV2):
test_loss                     : 0.882064
accuracy                      : 0.734167
balanced_accuracy             : 0.539963
precision                     : 0.766497
recall_sensitivity            : 0.928370
specificity                   : 0.151556
f1_score                      : 0.839704
mcc                           : 0.119972
roc_auc                       : 0.588467
pr_auc                        : 0.800671
average_precision             : 0.800692
eer                           : 0.437407
eer_threshold                 : 0.953869
false_positive_rate           : 0.848444
false_negative_rate           : 0.071630
true_negatives                : 682
false_positives               : 3818
false_negatives               : 967
true_positives                : 12533
number_of_test_images         : 18000


In [28]:
#celeb on dfc
print("\nTest results of DFC on Celeb-DF(V2) dataset (SwinV2):")
test_dataset = ImageClassificationDataset(test_celeb, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of DFC on Celeb-DF(V2) dataset (SwinV2):
test_loss                     : 0.582836
accuracy                      : 0.845160
balanced_accuracy             : 0.492399
precision                     : 0.902795
recall_sensitivity            : 0.928757
specificity                   : 0.056042
f1_score                      : 0.915592
mcc                           : -0.017559
roc_auc                       : 0.520440
pr_auc                        : 0.913680
average_precision             : 0.913716
eer                           : 0.475934
eer_threshold                 : 0.987292
false_positive_rate           : 0.943958
false_negative_rate           : 0.071243
true_negatives                : 32
false_positives               : 539
false_negatives               : 384
true_positives                : 5006
number_of_test_images         : 5961


In [30]:
#FF++ on hog
print("\nTest results of dfc on FF++ dataset (SwinV2):")
test_dataset = ImageClassificationDataset(test_ff, test_ff_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of dfc on FF++ dataset (SwinV2):
test_loss                     : 3.033166
accuracy                      : 0.427879
balanced_accuracy             : 0.495963
precision                     : 0.414406
recall_sensitivity            : 0.903977
specificity                   : 0.087950
f1_score                      : 0.568293
mcc                           : -0.013817
roc_auc                       : 0.505662
pr_auc                        : 0.424866
average_precision             : 0.425596
eer                           : 0.496086
eer_threshold                 : 0.995493
false_positive_rate           : 0.912050
false_negative_rate           : 0.096023
true_negatives                : 127
false_positives               : 1317
false_negatives               : 99
true_positives                : 932
number_of_test_images         : 2475


# FF++

LOAD THE DATASET

In [8]:
import os, cv2, numpy as np

FINAL_ROOT = r'D:\thesis\ff_final'

def load_split(split, cls):
    base = os.path.join(FINAL_ROOT, split, cls)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        d = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(d, f)) for f in sorted(os.listdir(d))]
        if frames:
            nested.append(frames); ids.append(vid_id)
    return nested, ids

# Main splits
ff_real_train_f, ff_real_train_ids = load_split('train', 'real')
ff_fake_train_f, ff_fake_train_ids = load_split('train', 'fake')
ff_real_val_f,   ff_real_val_ids   = load_split('val',   'real')
ff_fake_val_f,   ff_fake_val_ids   = load_split('val',   'fake')
ff_real_test_f,  ff_real_test_ids  = load_split('test',  'real')
ff_fake_test_f,  ff_fake_test_ids  = load_split('test',  'fake')

print("Reloaded main splits. Example shape:", np.shape(ff_real_train_f[0][0]))  # (160,160,3)
print("Real train videos:", len(ff_real_train_f), "| Fake train videos:", len(ff_fake_train_f))
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Combine all frames from the real and fake video groups.

    Labels:
        0 = Real
        1 = Fake
    """

    real_frames = [
        frame
        for video in real_videos
        for frame in video
        if frame is not None
    ]

    fake_frames = [
        frame
        for video in fake_videos
        for frame in video
        if frame is not None
    ]

    if not real_frames:
        raise ValueError("No real frames found.")

    if not fake_frames:
        raise ValueError("No fake frames found.")

    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training data
train_ff, train_ff_labels = combine_split(
    ff_real_train_f,
    ff_fake_train_f
)

# Validation data
val_ff, val_ff_labels = combine_split(
    ff_real_val_f,
    ff_fake_val_f
)

# Testing data
test_ff, test_ff_labels = combine_split(
    ff_real_test_f,
    ff_fake_test_f
)
print("\nTRAIN")
print("Images:", train_ff.shape)
print("Labels:", train_ff_labels.shape)
print("Real:", np.sum(train_ff_labels == 0))
print("Fake:", np.sum(train_ff_labels == 1))

print("\nVALIDATION")
print("Images:", val_ff.shape)
print("Labels:", val_ff_labels.shape)
print("Real:", np.sum(val_ff_labels == 0))
print("Fake:", np.sum(val_ff_labels == 1))

print("\nTEST")
print("Images:", test_ff.shape)
print("Labels:", test_ff_labels.shape)
print("Real:", np.sum(test_ff_labels == 0))
print("Fake:", np.sum(test_ff_labels == 1))

print("\nData types")
print("Train images:", train_ff.dtype)
print("Train labels:", train_ff_labels.dtype)

Reloaded main splits. Example shape: (160, 160, 3)
Real train videos: 517 | Fake train videos: 320

TRAIN
Images: (4595, 160, 160, 3)
Labels: (4595,)
Real: 2948
Fake: 1647

VALIDATION
Images: (948, 160, 160, 3)
Labels: (948,)
Real: 499
Fake: 449

TEST
Images: (2475, 160, 160, 3)
Labels: (2475,)
Real: 1444
Fake: 1031

Data types
Train images: uint8
Train labels: uint8


In [9]:
# Convert data into PyTorch Dataset
train_dataset = ImageClassificationDataset(train_ff, train_ff_labels, processor)
val_dataset = ImageClassificationDataset(val_ff, val_ff_labels, processor)
test_dataset = ImageClassificationDataset(test_ff, test_ff_labels, processor)

# Dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [10]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=10
)

Epoch 01/10 | Train Loss: 0.3591 | Train Accuracy: 0.8366 | Val Loss: 0.2335 | Val Accuracy: 0.8903
Epoch 02/10 | Train Loss: 0.2346 | Train Accuracy: 0.9045 | Val Loss: 0.1942 | Val Accuracy: 0.9314
Epoch 03/10 | Train Loss: 0.1609 | Train Accuracy: 0.9341 | Val Loss: 0.3046 | Val Accuracy: 0.8534
Epoch 04/10 | Train Loss: 0.1251 | Train Accuracy: 0.9476 | Val Loss: 0.1786 | Val Accuracy: 0.9251
Epoch 05/10 | Train Loss: 0.1102 | Train Accuracy: 0.9569 | Val Loss: 0.2459 | Val Accuracy: 0.9030
Epoch 06/10 | Train Loss: 0.1051 | Train Accuracy: 0.9602 | Val Loss: 0.2044 | Val Accuracy: 0.9103
Epoch 07/10 | Train Loss: 0.0903 | Train Accuracy: 0.9634 | Val Loss: 0.1957 | Val Accuracy: 0.9241
Epoch 08/10 | Train Loss: 0.0744 | Train Accuracy: 0.9695 | Val Loss: 0.2074 | Val Accuracy: 0.9420
Epoch 09/10 | Train Loss: 0.0660 | Train Accuracy: 0.9719 | Val Loss: 0.2960 | Val Accuracy: 0.8914
Epoch 10/10 | Train Loss: 0.0666 | Train Accuracy: 0.9754 | Val Loss: 0.3326 | Val Accuracy: 0.8977


In [12]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [11]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader
)
print("\nSWINV2-TINY TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )

SWINV2-TINY TEST RESULTS
test_loss                     : 0.297513
accuracy                      : 0.904242
balanced_accuracy             : 0.908366
precision                     : 0.851327
recall_sensitivity            : 0.933075
specificity                   : 0.883657
f1_score                      : 0.890329
mcc                           : 0.808336
roc_auc                       : 0.975536
pr_auc                        : 0.967583
average_precision             : 0.967520
eer                           : 0.094272
eer_threshold                 : 0.702540
false_positive_rate           : 0.116343
false_negative_rate           : 0.066925
true_negatives                : 1276
false_positives               : 168
false_negatives               : 69
true_positives                : 962
number_of_test_images         : 2475


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 9.2%
Time Usage: 36.0 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [15]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 11.8%
Time Usage: 38.2 s
GPU Memory Used: 440.5 MB
Power Consumption: 93W


#save the model

In [16]:
# ============================================================
# SAVE SWINV2 MODEL
# ============================================================

SAVE_DIR = (
    r"D:\thesis\results"
    r"\swinv2_tiny_ff_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


model.save_pretrained(
    SAVE_DIR
)

processor.save_pretrained(
    SAVE_DIR
)


torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,



        "label_mapping": {
            0: "real",
            1: "fake"
        }
    },

    os.path.join(
        SAVE_DIR,
        "training_checkpoint.pt"
    )
)


print("SwinV2 model saved to:")
print(SAVE_DIR)

SwinV2 model saved to:
D:\thesis\results\swinv2_tiny_ff_160


In [17]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_8208\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [18]:
# ============================================================
# PYTORCH SWINV2 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


BATCH_SIZE = 16
NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # Supports a dataset returning one dictionary:
        # {"pixel_values": ..., "labels": ...}
        if isinstance(batch, dict):

            pixel_values = batch[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        # Supports a dataset returning:
        # inputs_dictionary, labels
        else:

            inputs, labels = batch

            pixel_values = inputs[
                "pixel_values"
            ].to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

        outputs = model(
            pixel_values=pixel_values
        )

        logits = outputs.logits

        # Access one value to ensure output creation
        _ = logits[-1, 0]


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# FIVE REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting SwinV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for batch in test_loader:

            if isinstance(batch, dict):

                pixel_values = batch[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            else:

                inputs, labels = batch

                pixel_values = inputs[
                    "pixel_values"
                ].to(
                    device,
                    dtype=torch.float32,
                    non_blocking=True
                )

            outputs = model(
                pixel_values=pixel_values
            )

            logits = outputs.logits

            last_logits = logits

            processed_images += (
                pixel_values.size(0)
            )


    # CUDA execution is asynchronous
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del outputs


results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual SwinV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 2475
Test batches: 155

Performing warm-up using 3 batches...
Warm-up completed.

Starting SwinV2 resource run 1/5
Run 1: 31.41 seconds | 12.6923 ms/image | 78.79 images/s

Starting SwinV2 resource run 2/5
Run 2: 30.95 seconds | 12.5041 ms/image | 79.97 images/s

Starting SwinV2 resource run 3/5
Run 3: 31.35 seconds | 12.6679 ms/image | 78.94 images/s

Starting SwinV2 resource run 4/5
Run 4: 31.54 seconds | 12.7452 ms/image | 78.46 images/s

Starting SwinV2 resource run 5/5
Run 5: 31.32 seconds | 12.6543 ms/image | 79.02 images/s

Individual SwinV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,31.413364,12.692268,78.788125,3.119463,7.546875,5801.132812,5789.572368,5801.332031,0.000000,0.199219,...,526.385965,532.0,40.522807,99,32.616102,86.394,0.286519,1,2475,1.383244
1,30.947646,12.504100,79.973771,3.124632,6.718750,5785.906250,5789.900879,5793.601562,3.994629,7.695312,...,526.285714,532.0,40.782143,99,34.495064,97.913,0.298798,2,2475,1.383244
2,31.353088,12.667914,78.939594,3.081695,6.659375,5786.003906,5790.008271,5799.878906,4.004365,13.875000,...,526.306050,532.0,39.519573,82,35.445021,85.261,0.311357,3,2475,1.383244
3,31.544311,12.745176,78.461059,3.172320,6.753125,5786.019531,5789.993205,5800.718750,3.973674,14.699219,...,526.444444,532.0,40.041667,100,33.155910,87.381,0.291043,4,2475,1.383244
4,31.319294,12.654260,79.024769,3.099357,6.659375,5786.039062,5790.079773,5799.828125,4.040711,13.789062,...,526.326241,532.0,40.042553,100,33.326241,97.267,0.292032,5,2475,1.383244


In [19]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("Cross-Vit RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


Cross-Vit RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,31.315540,0.222865,31.038817,31.592264
1,latency_ms_per_image,12.652744,0.090046,12.540936,12.764551
2,throughput_images_per_s,79.037464,0.565868,78.334846,79.740081
3,average_cpu_percent,3.119493,0.034100,3.077153,3.161834
4,peak_cpu_percent,6.867500,0.381900,6.393308,7.341692
5,average_ram_mb,5789.910899,0.199673,5789.662972,5790.158826
6,peak_ram_mb,5799.071875,3.121219,5795.196370,5802.947380
7,average_incremental_ram_mb,3.202676,1.790514,0.979458,5.425893
8,peak_incremental_ram_mb,10.051562,6.180794,2.377093,17.726032
9,average_gpu_memory_mb,3243.519214,5.387198,3236.830124,3250.208304



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 12.653 ± 0.090 (95% CI: 12.541–12.765)
peak_ram_mb: 5799.072 ± 3.121 (95% CI: 5795.196–5802.947)
peak_gpu_memory_mb: 3249.170 ± 5.367 (95% CI: 3242.506–3255.833)
average_gpu_utilization_percent: 40.182 ± 0.488 (95% CI: 39.575–40.788)
average_gpu_power_w: 33.808 ± 1.143 (95% CI: 32.388–35.227)


#gernalization

In [21]:
#wild deepfake on ff
print("\nTest results of FF++ on wild deepfake dataset SwinV2:")
test_dataset = ImageClassificationDataset(test_images, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of FF++ on wild deepfake dataset SwinV2:
test_loss                     : 1.682252
accuracy                      : 0.349611
balanced_accuracy             : 0.490556
precision                     : 0.733403
recall_sensitivity            : 0.208667
specificity                   : 0.772444
f1_score                      : 0.324895
mcc                           : -0.019964
roc_auc                       : 0.500520
pr_auc                        : 0.740525
average_precision             : 0.740444
eer                           : 0.504815
eer_threshold                 : 0.127310
false_positive_rate           : 0.227556
false_negative_rate           : 0.791333
true_negatives                : 3476
false_positives               : 1024
false_negatives               : 10683
true_positives                : 2817
number_of_test_images         : 18000


In [23]:
#celeb on ff
print("\nTest results of FF++ on Celeb-df(v2) dataset (SwinV2):")
test_dataset = ImageClassificationDataset(test_celeb, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of FF++ on Celeb-df(v2) dataset (SwinV2):
test_loss                     : 1.751201
accuracy                      : 0.450428
balanced_accuracy             : 0.687492
precision                     : 0.994850
recall_sensitivity            : 0.394249
specificity                   : 0.980736
f1_score                      : 0.564709
mcc                           : 0.230149
roc_auc                       : 0.799703
pr_auc                        : 0.974660
average_precision             : 0.974663
eer                           : 0.289196
eer_threshold                 : 0.042905
false_positive_rate           : 0.019264
false_negative_rate           : 0.605751
true_negatives                : 560
false_positives               : 11
false_negatives               : 3265
true_positives                : 2125
number_of_test_images         : 5961


In [ ]:
#DFC on ff
print("\nTest results of FF++ on DFC dataset (SwinV2):")
test_dataset = ImageClassificationDataset(test_hog, test_labels, processor)
# Dataloaders
batch_size = 16
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
test_results = evaluate_complete(model=model,loader=test_loader)
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of FF++ on DFC dataset (SwinV2):
test_loss                     : 1.993874
accuracy                      : 0.439667
balanced_accuracy             : 0.439667
precision                     : 0.417200
recall_sensitivity            : 0.304000
specificity                   : 0.575333
f1_score                      : 0.351716
mcc                           : -0.125370
roc_auc                       : 0.404358
pr_auc                        : 0.427996
average_precision             : 0.428692
eer                           : 0.572000
eer_threshold                 : 0.207955
false_positive_rate           : 0.424667
false_negative_rate           : 0.696000
true_negatives                : 863
false_positives               : 637
false_negatives               : 1044
true_positives                : 456
number_of_test_images         : 3000


: 